In [ ]:
import os, re, unicodedata, warnings, textwrap, shutil
from collections import Counter
from datetime import datetime
from typing import List, Tuple, Optional

# ---------- Runtime / Threads ----------
os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")   # extra belt
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

import torch
torch.set_num_threads(12)
torch.set_num_interop_threads(3)

# Globálne vypni SDPA/flash jadra (kvôli 4-bit stabilite)
try:
    from torch.backends.cuda import sdp_kernel
    sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=False)
except Exception:
    pass

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import transformers
transformers.logging.set_verbosity_error()

from docx import Document
from striprtf.striprtf import rtf_to_text

# ---------- RAG (Chroma) ----------
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ---------- Graphics ----------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============ CONFIG ============

MODE = "title"  # 'title' alebo 'keyword'

INPUT_DIR   = "./Input"
OUTPUT_DIR  = "./Output"

THESAURUS_XLSX = "./Thesaurus/SK_Local_Thesaurus.xlsx"
THESAURUS_COL  = "slovak_terms"
STOPWORDS_TXT  = "./Input/stopword_dictionary.txt"

# Models (no reasoning)
SUMM_MODEL_ID   = "slovak-nlp/mistral-sk-7b"                  # causal LM -> summarization via prompt
GEN_MODEL_ID    = "nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1"     # instruction LM -> title/keyword

# Quantization & generation params
USE_4BIT_SUMM   = True
USE_4BIT_GEN    = True

SUMM_MAX_INPUT_CHARS = 8000
SUMM_ARGS_GEN = dict(max_new_tokens=96, do_sample=True, temperature=0.2, top_p=1.1)

GEN_BATCH       = 6
GEN_ARGS        = dict(max_new_tokens=96, do_sample=True, temperature=0.2, top_p=1.0, repetition_penalty=1.1)

# RAG / Embeddings
EMB_MODEL          = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
RAG_PERSIST_DIR    = "./RAG_store"
RAG_COLLECTION_DOC = "sk_legal_local_docs"
RAG_COLLECTION_TH  = "sk_legal_local_thesaurus"
RAG_TOP_K_DOC      = 3
RAG_TOP_K_TH       = 8

# Reports
GENERATE_RAG_REPORT = True
REPORT_TOP_TERMS    = 20
MAX_TEXT_CHARS      = 8000

# ============ Utils ============

def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m: return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]
    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"Thesaurus not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))
    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t: continue
        if len(t) == 1 or t.lower() in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    if not text: return []
    hits, t_na = {}, strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0: hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0]))) ]
    return ordered[:max_terms]

# ============ DOCX/RTF split ============

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None: buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None: buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# ============ RAG init (docs + thesaurus) ============

def _open_vector_store(collection_name: str, embeddings) -> Chroma:
    return Chroma(collection_name=collection_name, persist_directory=RAG_PERSIST_DIR, embedding_function=embeddings)

def _ensure_rag():
    os.makedirs(RAG_PERSIST_DIR, exist_ok=True)
    embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)

    def open_or_recreate(name: str):
        try:
            vs = _open_vector_store(name, embeddings)
            _ = vs._collection.get(limit=1)
            return vs
        except Exception as e:
            print(f"[RAG] Recreating '{name}' due to: {e}")
            try:
                Chroma(collection_name=name, persist_directory=RAG_PERSIST_DIR, embedding_function=embeddings)._collection.delete(where={})
            except Exception:
                pass
            return _open_vector_store(name, embeddings)

    vs_docs = open_or_recreate(RAG_COLLECTION_DOC)
    vs_th   = open_or_recreate(RAG_COLLECTION_TH)

    # Ensure thesaurus terms exist once
    try:
        existing = vs_th._collection.count()
        if existing == 0 and THESAURUS_TERMS:
            texts = [t for t in THESAURUS_TERMS]
            metas = [{"type": "th_term", "idx": i} for i in range(len(texts))]
            vs_th.add_texts(texts=texts, metadatas=metas, ids=[f"th_{i}" for i in range(len(texts))])
            print(f"[RAG] Added {len(texts)} thesaurus terms to vector store.")
    except Exception as e:
        print(f"[RAG] Thesaurus upsert failed: {e}")

    return vs_docs, vs_th, embeddings

VS_DOCS, VS_TH, EMBEDDINGS = _ensure_rag()

def rag_upsert_docs(doc_id: str, texts: List[str], metas: List[dict]):
    if VS_DOCS is None or not texts: return
    try:
        VS_DOCS.add_texts(texts=texts, metadatas=metas, ids=[f"{doc_id}_{i}" for i in range(len(texts))])
    except Exception as e:
        print(f"[RAG] docs upsert failed: {e}")

def rag_retrieve(summary: str):
    docs, th = [], []
    try:
        if VS_DOCS is not None and summary:
            d = VS_DOCS.similarity_search(summary, k=RAG_TOP_K_DOC)
            docs = [x.page_content for x in d]
    except Exception as e:
        print(f"[RAG] docs retrieve failed: {e}")
    try:
        if VS_TH is not None and summary:
            d = VS_TH.similarity_search(summary, k=RAG_TOP_K_TH)
            th  = [x.page_content for x in d]
    except Exception as e:
        print(f"[RAG] thesaurus retrieve failed: {e}")
    return docs, th

# ============ Model loaders (both causal) ============

def _qconf_4bit():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

def _post_configure_model(model):
    # Stabilita: eager attention, cache on, no grad checkpointing
    try:
        model.config._attn_implementation = "eager"
    except Exception:
        pass
    try:
        model.config.use_cache = True
        if hasattr(model, "gradient_checkpointing_disable"):
            model.gradient_checkpointing_disable()
    except Exception:
        pass
    return model

def _load_causal_lm(model_id: str, use_4bit: bool):
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    qconf = _qconf_4bit() if use_4bit else None
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device_map,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconf,
        torch_dtype=None if qconf else torch.float16
    )
    model = _post_configure_model(model)
    return pipeline("text-generation",model=model,tokenizer=tok,device_map=device_map,return_full_text=False,)

# Summarizer (causal LM by prompt)
SUMM_PIPE = _load_causal_lm(SUMM_MODEL_ID, USE_4BIT_SUMM)
print("[SUMM] Loaded:", SUMM_MODEL_ID)

# Generator (title/keyword)
GEN_PIPE  = _load_causal_lm(GEN_MODEL_ID, USE_4BIT_GEN)
print("[GEN ] Loaded:", GEN_MODEL_ID)

# Lazy CPU fallback pre GEN
_GEN_PIPE_CPU = None
def _ensure_gen_cpu():
    global _GEN_PIPE_CPU
    if _GEN_PIPE_CPU is not None:
        return _GEN_PIPE_CPU
    tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_ID,
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float32
    )
    model = _post_configure_model(model)
    _GEN_PIPE_CPU = pipeline("text-generation", model=model, tokenizer=tok, device_map="cpu")
    return _GEN_PIPE_CPU

# ============ Stage 1: Summarization ============

def _truncate(s: str, n: int) -> str:
    s = (s or "").strip()
    return s if len(s) <= n else s[:n]

def summarize_batch(texts: List[str]) -> List[str]:
    prompts = [
        ("Zhrň nasledujúci text v slovenčine do 1–5 viet, maximálne ~100 slov."
            "\nTEXT:\n" + _truncate(t, SUMM_MAX_INPUT_CHARS) + "\nZHRNUTIE:")
        for t in texts
    ]
    outs = SUMM_PIPE(prompts, **SUMM_ARGS_GEN, batch_size=min(GEN_BATCH, len(prompts)))

    summaries = []
    for o in outs:
        s = o[0]["generated_text"] if isinstance(o, list) else (o.get("generated_text") or o.get("text") or "")
        s = (s or "").strip()

        if "ZHRNUTIE:" in s:
            s = s.split("ZHRNUTIE:")[-1]

        s = re.sub(r'^.*?ZHRNUTIE:\s*', '', s, flags=re.DOTALL)

        # 3) finálne vyčistenie
        s = re.sub(r'\s+', ' ', s).strip()
        summaries.append(s)
    return summaries


# ============ Stage 2: From summaries to title/keyword ============

def prompt_from_summary(mode: str, summary: str, th_terms: List[str], doc_ctx: List[str]) -> str:
    th_list  = " • " + "\n • ".join(th_terms[:12]) if th_terms else "(bez zhôd)"
    doc_join = ("\nKONTEXT:\n" + "\n---\n".join(doc_ctx[:2])) if doc_ctx else ""
    if mode == "title":
        return (
            "ÚLOHA: Na základe sumarizácie vytvor JEDEN presný slovenský právny nadpis.\n"
            "PRAVIDLÁ: 3–10 slov • bez bodky • bez úvodzoviek • jeden riadok • vecný.\n"
            f"Relevantné pojmy (tezaurus):\n{th_list}\n\n"
            f"SUMARIZÁCIA:\n{summary}\n"
            f"{doc_join}\n\n"
            "NADPIS:"
        )
    else:
        return (
            "ÚLOHA: Z nasledujúcej sumarizácie vyber JEDEN najvhodnejší slovenský právny pojem (1–3 slová).\n"
            "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky. Jeden riadok.\n"
            f"Relevantné pojmy (tezaurus): {', '.join(th_terms[:12])}\n\n"
            f"SUMARIZÁCIA:\n{summary}\n"
            f"{doc_join}\n\n"
            "POJEM:"
        )

def _normalize_gen_item(item) -> str:
    raw = ""
    if isinstance(item, list) and item and isinstance(item[0], dict):
        raw = item[0].get("generated_text") or ""
    elif isinstance(item, dict):
        raw = item.get("generated_text") or ""
    else:
        raw = str(item or "")
    txt = raw.strip()
    txt = re.sub(r'^[\-\•\:\s]+', '', txt)
    txt = re.sub(r'[\.，。…]+$', '', txt)
    txt = txt.strip(' "„”')
    return txt.splitlines()[0].strip() if txt else ""

def _gen_safe(prompt: str) -> str:
    """Try GPU once; on 'illegal memory access' (or similar) fall back to CPU."""
    try:
        out = GEN_PIPE(prompt, **GEN_ARGS, batch_size=1)
        return _normalize_gen_item(out)
    except RuntimeError as e:
        msg = str(e)
        if ("illegal memory access" in msg) or ("CUBLAS_STATUS_EXECUTION_FAILED" in msg):
            # Reset CUDA allocator state and try CPU
            try:
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            except Exception:
                pass
            cpu_pipe = _ensure_gen_cpu()
            out = cpu_pipe(prompt, **GEN_ARGS, batch_size=1)
            return _normalize_gen_item(out)
        raise

def generate_from_summaries(mode: str, summaries: List[str], prio_terms_list: List[List[str]]):
    outputs, rag_terms_per_section, rag_docs_per_section = [], [], []
    for i, summary in enumerate(summaries):
        doc_ctx, th_ctx = rag_retrieve(summary)
        rag_terms_per_section.append(th_ctx)
        rag_docs_per_section.append(doc_ctx)
        merged_terms = list(dict.fromkeys((prio_terms_list[i] or []) + th_ctx))  # order-preserving de-dup

        prompt = prompt_from_summary(mode, summary, merged_terms, doc_ctx)
        gen = _gen_safe(prompt)
        outputs.append(gen)
    return outputs, rag_terms_per_section, rag_docs_per_section

# ============ Graphic RAG report (from summaries) ============

def _wrap(s, width=46): return "\n".join(textwrap.wrap(s, width=width)) if s else s

def build_graphic_report(
    mode: str,
    files: List[str],
    headers: List[str],
    summaries: List[str],
    rag_terms_per_section: List[List[str]],
    rag_docs_per_section: List[List[str]],
    outputs: List[str],
    out_dir: str
):
    import os
    os.makedirs(out_dir, exist_ok=True)
    # 1) Top thesaurus terms across sections
    flat_terms = [t for lst in rag_terms_per_section for t in lst]
    term_counts = Counter([t.strip().lower() for t in flat_terms if t.strip()])
    top_terms = term_counts.most_common(REPORT_TOP_TERMS)

    # 2) Coverage per section
    coverage = [len(set([t.strip().lower() for t in lst])) for lst in rag_terms_per_section]

    # Plot
    fig_h = 6 + max(0, len(top_terms)-12)*0.18
    fig = plt.figure(figsize=(12, fig_h))
    gs = fig.add_gridspec(2, 1, height_ratios=[2, 1.2], hspace=0.35)

    # Bar chart: top terms
    ax1 = fig.add_subplot(gs[0, 0])
    if top_terms:
        labels, vals = zip(*top_terms)
        y_pos = range(len(labels))
        ax1.barh(y_pos, vals)
        ax1.set_yticks(list(y_pos))
        ax1.set_yticklabels([_wrap(l, 36) for l in labels])
        ax1.invert_yaxis()
        ax1.set_xlabel("Počet výskytov v RAG (tezaurus)")
        ax1.set_title("Top RAG tezaurus pojmy (zo sumarizácií)")
    else:
        ax1.text(0.5, 0.5, "Žiadne termíny", ha="center", va="center")
        ax1.set_axis_off()

    # Line: coverage per section
    ax2 = fig.add_subplot(gs[1, 0])
    xs = list(range(1, len(coverage)+1))
    ax2.plot(xs, coverage, marker="o", linewidth=2)
    ax2.set_xticks(xs)
    ax2.set_xlabel("Sekcia (index)")
    ax2.set_ylabel("Počet pojmov z RAG")
    ax2.set_title("Pokrytie RAG pojmami na sekciu")

    today = datetime.today().strftime("%Y-%m-%d")
    png_path = os.path.join(out_dir, f"rag_graph_report_{mode}_{today}.png")
    fig.tight_layout()
    fig.savefig(png_path, dpi=150)
    plt.close(fig)

    # Detail CSV
    rows = []
    for i in range(len(files)):
        rows.append({
            "file": files[i],
            "header": headers[i],
            "summary": summaries[i],
            "rag_th_terms": "; ".join(rag_terms_per_section[i]),
            "rag_doc_snippets": " | ".join([d.replace("\n", " ")[:240] for d in rag_docs_per_section[i]]),
            ("suggested_title" if mode=="title" else "top_keyword"): outputs[i]
        })
    csv_path = os.path.join(out_dir, f"rag_graph_report_detail_{mode}_{today}.csv")
    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"[REPORT] Saved graphic: {png_path}")
    print(f"[REPORT] Saved detail CSV: {csv_path}")
    return png_path, csv_path

# ============ DRIVER ============

def process_all(input_dir=INPUT_DIR):
    files = sorted(os.listdir(input_dir), key=str.lower)
    jobs = []  # (mode, file, header, full_text, prio_terms)
    sec_files, sec_headers = [], []

    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith(".docx"):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith(".rtf"):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f"No text in {f}"); continue

        # Upsert to DOCS store for RAG
        section_texts, metas = [], []
        for idx, (header, text) in enumerate(secs):
            if header == "__DOCUMENT__": continue
            if text.strip():
                section_texts.append(text.strip())
                metas.append({"file": f, "header": header, "idx": idx})
        if section_texts:
            try:
                rag_upsert_docs(os.path.splitext(f)[0], section_texts, metas)
            except Exception as e:
                print(f"[RAG] upsert skip ({f}): {e}")

        for header, text in secs:
            if not text.strip(): continue
            full = text if len(text) <= MAX_TEXT_CHARS else text[:MAX_TEXT_CHARS]
            prio_terms = terms_matched_in_text(full, max_terms=30)
            jobs.append((MODE, f, header, full, prio_terms))
            sec_files.append(f)
            sec_headers.append(header)

        print(f"Queued {f} with {len(secs)} sections.")

    if not jobs:
        print("No jobs queued.")
        return pd.DataFrame([])

    # Stage 1: summarize
    texts = [j[3] for j in jobs]
    summaries = summarize_batch(texts)

    # Stage 2: generate from summaries (GPU→CPU safe)
    prio_terms_list = [j[4] for j in jobs]
    gens, rag_terms_per_section, rag_docs_per_section = generate_from_summaries(MODE, summaries, prio_terms_list)

    # Export main CSV
    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    rows = []
    for i, ((mode, f, header, _full, prio), gen) in enumerate(zip(jobs, gens)):
        if mode == "title":
            rows.append({
                "file": f, "header": header,
                "summary": summaries[i],
                "suggested_title": gen,
                "method": f"gen-summarize({SUMM_MODEL_ID}) + RAG + single-gen ({GEN_MODEL_ID})",
                "priority_terms": "; ".join(prio[:20])
            })
        else:
            rows.append({
                "file": f, "header": header,
                "summary": summaries[i],
                "top_keyword": gen,
                "method": f"gen-summarize({SUMM_MODEL_ID}) + RAG + single-gen ({GEN_MODEL_ID})",
                "priority_terms": "; ".join(prio[:20])
            })
    stub = "titles_from_summary" if MODE=="title" else "keywords_from_summary"
    csv_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")

    # Graphic report
    if GENERATE_RAG_REPORT:
        try:
            build_graphic_report(
                MODE, sec_files, sec_headers, summaries,
                rag_terms_per_section, rag_docs_per_section, gens, OUTPUT_DIR
            )
        except Exception as e:
            print(f"[REPORT] Failed to build graphic report: {e}")

    return pd.DataFrame(rows)

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=r".*Some weights.*not of the same dtype.*")
    warnings.filterwarnings("ignore", message=r".*Tokenizer parallelism.*")

    df = process_all(INPUT_DIR)
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))


c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
C:\Users\nyx3t\AppData\Local\Temp\ipykernel_39188\3712925558.py:254: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMB_MODEL)
c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered inte

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[SUMM] Loaded: slovak-nlp/mistral-sk-7b


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[GEN ] Loaded: nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1
Queued 117694.docx with 8 sections.
Queued 117695.docx with 2 sections.
Queued 117696.docx with 11 sections.
Queued 117697.docx with 15 sections.
Queued 117698.docx with 8 sections.
Queued 117699.docx with 18 sections.
Queued 117700.docx with 1 sections.
Queued 117701.docx with 6 sections.
Queued 117702.docx with 12 sections.
Queued 117703.docx with 10 sections.
Saved: ./Output\titles_from_summary_2025-11-17.csv


C:\Users\nyx3t\AppData\Local\Temp\ipykernel_39188\3712925558.py:533: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[REPORT] Saved graphic: ./Output\rag_graph_report_title_2025-11-17.png
[REPORT] Saved detail CSV: ./Output\rag_graph_report_detail_title_2025-11-17.csv


,file,header,summary,suggested_title,method,priority_terms
0,117694.docx,__DOCUMENT__,Vyporiadanie BSM v_dedičskom konaní je vždy vy...,Vyporiadanie BSM v dedičskem konaní („napríkla...,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,podiel; dohoda; konanie (štádium procesného ko...
1,117694.docx,po-heading-id_EHU0iUd9Y0yOqqj7VGbM1A,V prípade podielov v podielovom spoluvlastníct...,Ústavnou rovnakú rozpisu prevodné zmluvy s úst...,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,podiel; vyporiadanie; spoločník; autor
2,117694.docx,po-heading-id_c6ihO-fo_kmbMqy9OwPZ5g,"Najvyšší súd sa v rozhodnutí zaoberal otázkou,",0.0,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,uznesenie
3,117694.docx,po-heading-id_aWeISBCNy0mMqxGjUkaTNw,"Najvyšší súd vyslovil, že súd v_konaní o",NAFONYTANIE NA KONVÈRTUALNYCH ZASENI,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,otázka
4,117694.docx,po-heading-id_VtH6pbc6j0qtnD6ytAcu1w,Vyporiadanie BSM v_dedičskom konaní je v_záujme š,NAPISANÉT FAŠOBILO V ZASECHNOSTI,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,konanie (štádium procesného konania); majetok ...
5,117694.docx,po-heading-id_aWuBOYjuME-jlXoxHWEcHg,1.2Dohoda o vyporiadaní BSM a dedičská dohoda,Účelenie do hřodiny (summarium),gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,dohoda; právo; majetok (imanie); pohľadávky; m...
6,117694.docx,po-heading-id_-ctrw_j88Uazc2-86wvAAQ,1. § 150 OZ je kogentný a je teda nevyhnutné,ÚZDOVORUJEM NA KOGENTNOSTY OCHODNIKA O VYPOR,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,dohoda; úschova notárska (súdna); majetok (ima...
7,117694.docx,po-heading-id_kizOD8s8zEyXvfzM_OhAwg,"Záverom možno konštatovať, že vyporiadanie BSM...",NADPH (neodlaganská pravidla),gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,zmier; vyporiadanie; uznesenie; majetok (imani...
8,117695.docx,__DOCUMENT__,Všeobecné pravidlo delenia dôkazného bremena j...,PRIMÉRY PROJEKT - PRIMÉRY PROJEKT,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,bremeno; skutočnosti; norma; žalobca; dlžník; ...
9,117695.docx,po-heading-id_1OMtQ1Gs9kWEadIcZhz9ig,Všeobecné pravidlo delenia dôkazného bremena j...,PRIMÉRY PROJEKT - PRIMÉRY PROJEKT,gen-summarize(slovak-nlp/mistral-sk-7b) + RAG ...,bremeno; skutočnosti; norma; žalobca; dlžník; ...
